In [2]:
import pandas as pd


In [3]:
df = pd.read_csv("IMDB Dataset.csv")

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

# Pre-processing

In [7]:

# 1. Converting to lowercase

df["review"] = df["review"].str.lower()

In [8]:

# Example (not part of code)

import re
sample_text = "abc is the word , abc" # => abc = xyz
new_text = re.sub("abc", "xyz", sample_text)
print(new_text)

xyz is the word , xyz


In [9]:
# 2 Removing the URLs
import re

def remove_urls(text):
    text = re.sub(r"http\S+", "", text) # (pattern, repl, string) eg - https://www.google.com
    return text
    
df["review"] = df["review"].apply(remove_urls)


In [10]:
# 3 remove punctuations
def remove_punctuation (text):
    text = re.sub(r"[^A-Za-z0-9\s]","", text) # A-Z a-z 0-9 ls
    return text

df["review"] = df["review"].apply(remove_punctuation)

In [11]:
# 4. Remove HTML

def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [12]:
# 5. removing the stop-words

import nltk # sentence tokanizer

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [14]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    
    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
            
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [15]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [16]:
# 6 stemming
# running -> run
# played = play
# we use algo PorterStremming

from nltk.stem import PorterStemmer

In [17]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)

df["review"]=df["review"].apply(stemming)

In [18]:
# 7 Encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [19]:
y = df["sentiment"]

In [20]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int32

In [21]:
df

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1
...,...,...
49995,thought th move dd rght good job t wsnt s cret...,1
49996,bd plot bd dlogu bd ctng dotc drectng nnoyng p...,0
49997,ctholc tught n prochl elentri school nun tught...,0
49998,im gog dgree previou comnt side mlt e secd rte...,0


In [22]:
# 8 Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

x = tf.fit_transform(df["review"])

In [23]:
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

## Dataset and Data loaders

In [24]:
from sklearn.model_selection import train_test_split

X_train ,X_test, y_train , y_test = train_test_split(
    x , y , test_size=0.2 , random_state=42
)

In [25]:
X_train.shape

(39665, 5000)

In [26]:
import torch 
from torch.utils.data import TensorDataset, DataLoader

In [28]:
X_train = X_train.toarray()
X_test = X_test.toarray()


In [29]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [30]:
train_loader = DataLoader(train_set , shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [33]:
import torch.nn as nn
import torch.optim as optim

In [34]:
class RNN(nn.Module):
    def __init__(self , input_size , hidden_size = 128, num_layers=1):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # RNN layer 
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        
        # Fully connected layer 
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward (self, x):
        # optional => shape (num of layers, batch size , hiddden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        out, _ = self.rnn(x ,h0)
        # 1st values => hiddden stateof all the timesteps => (batch, seq_len, hiddden size)
        # 2nd values => final hidden state of last timestamp
        
        out = self.fc (out[:, -1, :])
        return out
        

In [35]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters())

In [36]:
# Traing the RNN

epochs = 10

for epoch in range(epochs):
    model.train()
    
    for xb , yb in train_loader:
        optimizer.zero_grad()
        
        xb = xb.unsqueeze(1) # add singleton direction
        
        outputs = model(xb) # (batch_size , 1)
        
        outputs = torch.sigmoid(outputs.squeeze()) #(batch_size), => probability
        
        loss = criterion(outputs,yb)
        loss.backward() # backprop
        optimizer.step() # weights updates
        
    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.3165542483329773
epoch = 2/10 and loss = 0.2798592448234558
epoch = 3/10 and loss = 0.19889409840106964
epoch = 4/10 and loss = 0.28904035687446594
epoch = 5/10 and loss = 0.1474602073431015
epoch = 6/10 and loss = 0.25771087408065796
epoch = 7/10 and loss = 0.23612403869628906
epoch = 8/10 and loss = 0.21164651215076447
epoch = 9/10 and loss = 0.2648732662200928
epoch = 10/10 and loss = 0.33368992805480957


In [37]:
# evaluate

model.eval()

with torch.no_grad():
    correct_val = 0
    tot_val = 0
    
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        
        outputs = model(xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        
        tot_val += yb.size(0)
        correct_val += (predicted == yb ).sum().item()
        
    print(f"Accuracy = {correct_val/tot_val*100}")

Accuracy = 85.63073510134113
